Temporal market trends and affordability indicators
Decoupled from `main.ipynb` on purpose (see Section 3.11 of the report) — this notebook reads
the CSVs `main.ipynb` already exports (Section 12: Export Outputs) rather than re-running the
pipeline, so visualisation work here can't be affected by changes to the backend model.

Three outputs, matching the three views scoped for Section 3.11:
1. Actual vs. predicted price trend by suburb over time (Temporal Market Trends)
2. Price-to-income affordability ratio by suburb (Affordability Indicator Distribution)
3. Random Forest vs. XGBoost feature importance (Feature Importance Charts)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# Adjust these to match where main.ipynb's OUTPUT_DIR / MODEL_READY_DIR actually point on your machine
OUTPUT_DIR = Path("../outputs")
MODEL_READY_DIR = Path("../data/model_ready")

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.csv"
MODEL_READY_PATH = MODEL_READY_DIR / "model_ready.csv"

BLUE = "#2E5090"

Temporal Market Trends (Actual VS Predicted price by suburb )

In [ ]:
test_output = pd.read_csv(PREDICTIONS_PATH, parse_dates=["sold_date_iso"])
test_output["sale_year"] = test_output["sold_date_iso"].dt.year

# predicted_<model> columns come straight from Section 12's export in main.ipynb
pred_cols = [c for c in test_output.columns if c.startswith("predicted_")]
print("Available model prediction columns:", pred_cols)

TOP_N_SUBURBS = 5
MODEL_COL = "predicted_xgboost"   # change to match one of the columns printed above

top_suburbs = test_output["suburb"].value_counts().head(TOP_N_SUBURBS).index
trend_df = test_output[test_output["suburb"].isin(top_suburbs)]

trend_summary = (
    trend_df.groupby(["suburb", "sale_year"])[["price_numeric", MODEL_COL]]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 6))
for suburb, grp in trend_summary.groupby("suburb"):
    grp = grp.sort_values("sale_year")
    ax.plot(grp["sale_year"], grp["price_numeric"], marker="o", label=f"{suburb} — Actual")
    ax.plot(grp["sale_year"], grp[MODEL_COL], marker="o", linestyle="--",
            label=f"{suburb} — Predicted")

model_label = MODEL_COL.replace("predicted_", "").replace("_", " ").title()
ax.set_title(f"Actual vs. Predicted Median Sale Price by Suburb Over Time ({model_label})")
ax.set_xlabel("Sale Year")
ax.set_ylabel("Mean Price (Test Set, AUD)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_temporal_market_trends.png", dpi=150, bbox_inches="tight")
plt.show()

Affordability indicator distribution (Price to Income ratio by suburb)

In [ ]:
model_ready = pd.read_csv(MODEL_READY_PATH)

# The income column's exact name depends on which sub-account the correlation-clustering
# step in main.ipynb (Section 2.3) picked -- list the candidates rather than hardcoding one.
income_candidates = [c for c in model_ready.columns if "income" in c.lower()]
print("Candidate income columns:", income_candidates)

INCOME_COL = income_candidates[0]   # change the index/name if this isn't the right one

afford = model_ready.groupby("suburb").agg(
    median_price=("price_numeric", "median"),
    income=(INCOME_COL, "median"),
).dropna()
afford["price_to_income_ratio"] = afford["median_price"] / afford["income"]
afford = afford.sort_values("price_to_income_ratio", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=afford["price_to_income_ratio"], y=afford.index, color=BLUE, ax=ax)
ax.set_title("Price-to-Income Ratio by Suburb (Top 15 Least Affordable)")
ax.set_xlabel("Median Sale Price / Median Per-Capita Income")
ax.set_ylabel("Suburb")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_affordability_ratio.png", dpi=150, bbox_inches="tight")
plt.show()